# Trash YOLO training

Runs on Colab (GPU). Steps:
1. Clone repo
2. Install deps
3. Load Kaggle credentials
4. Prep TACO dataset
5. Prep Kaggle Drink Waste dataset (optional)
6. Train YOLOv8n
7. Evaluate
8. Save weights to Drive

**Before running:** set `REPO_URL` below to your GitHub URL, and add `KAGGLE_USERNAME` + `KAGGLE_KEY` as Colab secrets (🔑 icon in left sidebar, toggle 'Notebook access').

In [ ]:
# --- config ---
REPO_URL = "https://github.com/Ndgandhi23/RU-IEEE-Hackathon.git"
REPO_DIR = "/content/RU-IEEE-Hackathon"
ML_DIR = f"{REPO_DIR}/ml-training"
DRIVE_OUT = "/content/drive/MyDrive/trash-yolo"  # weights will be copied here at the end

In [ ]:
# --- mount Drive (for saving weights at the end) ---
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# --- clone repo ---
import os, subprocess
if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=True)
os.chdir(ML_DIR)
print('cwd:', os.getcwd())

In [ ]:
# --- install deps ---
!pip install -q -r requirements.txt

In [ ]:
# --- Kaggle credentials (from Colab secrets) ---
from google.colab import userdata
import os
os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')
!kaggle --version

In [ ]:
# --- prep TACO (downloads ~1500 images, a few minutes) ---
!python scripts/prepare_dataset.py

In [ ]:
# --- prep Kaggle Drink Waste (~9k images, appends to splits) ---
!python scripts/prepare_drink_waste.py

In [ ]:
# --- quick sanity check on the dataset ---
from pathlib import Path
for split in ('train', 'val', 'test'):
    n = len(list(Path(f'data/images/{split}').glob('*')))
    print(f'{split}: {n} images')

In [ ]:
# --- train ---
!python scripts/train.py --epochs 100 --batch 32 --name trash_v1

In [ ]:
# --- evaluate on test split ---
!python scripts/evaluate.py runs/train/trash_v1/weights/best.pt --split test

In [ ]:
# --- save weights + run artifacts to Drive ---
import shutil, os
os.makedirs(DRIVE_OUT, exist_ok=True)
src_best = 'runs/train/trash_v1/weights/best.pt'
src_last = 'runs/train/trash_v1/weights/last.pt'
shutil.copy2(src_best, f'{DRIVE_OUT}/trash_v1_best.pt')
shutil.copy2(src_last, f'{DRIVE_OUT}/trash_v1_last.pt')
shutil.copytree('runs/train/trash_v1', f'{DRIVE_OUT}/trash_v1_run', dirs_exist_ok=True)
print('saved to', DRIVE_OUT)